In [ ]:
#API Produção Folha de Feijão

!pip install -q pyngrok flask flask_ngrok
!ngrok authtoken 2iSxfrOXy3MxIZFbyXTcHq4G8pZ_4oXz4n8jAS9kwtgwSG7QV

from flask import Flask, request, jsonify, render_template_string
from flask_ngrok import run_with_ngrok
from pyngrok import ngrok
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array
from PIL import Image
import io
import base64

from google.colab import drive
drive.mount('/content/drive')

# Configuração do Flask com Ngrok
app = Flask(__name__)

public_url = ngrok.connect(5000).public_url
print(f" * Ngrok Tunnel: {public_url}")
run_with_ngrok(app)

# Classe do Modelo
class BeanLeafClassifier:
    def __init__(self, model_path):
        try:
            # Suprimir warnings do TensorFlow
            import tensorflow as tf
            tf.get_logger().setLevel('ERROR')

            self.model = load_model(model_path)
            # Compilar o modelo para evitar warnings
            self.model.compile(optimizer='adam',
                            loss='categorical_crossentropy',
                            metrics=['accuracy'])
            self.class_names = ['Mancha_angular', 'Ferrugem', 'Saudavel']
            print("✅ Modelo carregado e compilado com sucesso!")
        except Exception as e:
            print(f"❌ Erro ao carregar modelo: {str(e)}")
            raise

    def predict(self, image_bytes):
        try:
            img = Image.open(io.BytesIO(image_bytes))
            img = img.convert('RGB').resize((224, 224))
            img_array = img_to_array(img) / 255.0
            img_array = np.expand_dims(img_array, axis=0)

            predictions = self.model.predict(img_array)
            predicted_idx = np.argmax(predictions[0])

            return {
                'class': self.class_names[predicted_idx],
                'confidence': round(float(np.max(predictions[0])) * 100, 2),
                'probabilities': {
                    cls: round(float(pred) * 100, 2)
                    for cls, pred in zip(self.class_names, predictions[0])
                },
                'image_base64': base64.b64encode(image_bytes).decode('utf-8')
            }
        except Exception as e:
            print(f"⚠️ Erro na predição: {str(e)}")
            raise

classifier = BeanLeafClassifier('/content/drive/MyDrive/modelo_folhas_sem_aug.h5')

HTML_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <title>Classificador de Folhas</title>
    <style>
        body { font-family: Arial, sans-serif; max-width: 800px; margin: 0 auto; padding: 20px; }
        .result-container { display: flex; margin-top: 20px; gap: 20px; }
        .image-preview { flex: 1; text-align: center; }
        .prediction-result { flex: 1; }
        .confidence-bar { height: 20px; background: #f0f0f0; border-radius: 10px; margin: 5px 0; }
        .confidence-fill { height: 100%; border-radius: 10px; background: #4CAF50; }
        .pred-class { font-size: 1.5em; color: #2E7D32; font-weight: bold; }
        .prob-item { margin: 10px 0; }
    </style>
</head>
<body>
    <h1>Classificador de Doenças em Folhas</h1>
    <form method="post" enctype="multipart/form-data">
        <input type="file" name="file" accept="image/*" required>
        <button type="submit">Classificar</button>
    </form>

    {% if result %}
    <div class="result-container">
        <div class="image-preview">
            <h3>Imagem Analisada</h3>
            <img src="data:image/jpeg;base64,{{ result.image_base64 }}" style="max-width: 100%;">
        </div>
        <div class="prediction-result">
            <h2>Resultado:</h2>
            <p class="pred-class">{{ result.class }} ({{ result.confidence }}%)</p>

            <h3>Probabilidades:</h3>
            {% for class_name, prob in result.probabilities.items() %}
            <div class="prob-item">
                <div>{{ class_name }}: {{ prob }}%</div>
                <div class="confidence-bar">
                    <div class="confidence-fill" style="width: {{ prob }}%"></div>
                </div>
            </div>
            {% endfor %}
        </div>
    </div>
    {% endif %}
</body>
</html>
"""

@app.route('/', methods=['GET', 'POST'])
def home():
    if request.method == 'POST':
        if 'file' not in request.files:
            return "Nenhum arquivo enviado", 400

        file = request.files['file']
        if file.filename == '':
            return "Nenhum arquivo selecionado", 400

        try:
            image_bytes = file.read()
            result = classifier.predict(image_bytes)
            return render_template_string(HTML_TEMPLATE, result=result)
        except Exception as e:
            return f"Erro ao processar imagem: {str(e)}", 500

    return render_template_string(HTML_TEMPLATE, result=None)

if __name__ == '__main__':
    app.run()

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Mounted at /content/drive
 * Ngrok Tunnel: https://3935-35-229-178-43.ngrok-free.app


✅ Modelo carregado e compilado com sucesso!
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [20/May/2025 18:26:29] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/May/2025 18:26:30] "GET /favicon.ico HTTP/1.1" 404 -


 * Running on http://3935-35-229-178-43.ngrok-free.app
 * Traffic stats available on http://127.0.0.1:4040
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


INFO:werkzeug:127.0.0.1 - - [20/May/2025 18:27:14] "POST / HTTP/1.1" 200 -


...

INFO:werkzeug:127.0.0.1 - - [20/May/2025 18:30:08] "POST / HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step


INFO:werkzeug:127.0.0.1 - - [20/May/2025 18:30:20] "POST / HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step


INFO:werkzeug:127.0.0.1 - - [20/May/2025 18:30:35] "POST / HTTP/1.1" 200 -


In [ ]:
# Carregar modelo p/ ver os parâmetros
from tensorflow import keras
from tensorflow.keras.models import load_model
from google.colab import drive
drive.mount('/content/drive')

model_carregado = keras.models.load_model('/content/drive/MyDrive/modelo_folha_feijao_04.h5')
model_carregado.summary()